# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)

On Colab this clones the repo and moves into it. Locally it just walks up to the repo root. Run this first — every other cell depends on it.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/vikraamkumar-ds/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found -- are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-internship-ml
Starter data found. You're ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **scoring (ranking)**. Each page gets a priority score, and the reviewer works down the list from highest to lowest score — because they only have time for the top 50 pages this cycle, not all 30,000 in the inventory. A plain yes/no classification ("needs attention" or not) is a useful building block, but on its own it isn't enough: too many pages would qualify as "needs attention" at once (see the code cell below), and the reviewer needs an *order* to work through, not just a flag. So under the hood I train a classifier (declining vs. not), and I use its predicted probability as the ranking score — the task *type* is scoring, the *mechanism* is classification.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

total_pages = len(df)
reviewer_capacity = 50
print(f"Total pages: {total_pages:,}")
print(f"Reviewer capacity this cycle: {reviewer_capacity}")
print(f"That's only {100*reviewer_capacity/total_pages:.2f}% of the inventory -- "
      f"order matters, not just a yes/no flag.")

Total pages: 30,000
Reviewer capacity this cycle: 50
That's only 0.17% of the inventory -- order matters, not just a yes/no flag.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is a **proxy**, not a perfect ground truth: whether a page is currently labeled as declining (`trend_direction == "down"`). It's defined by a rule applied to the trailing-90-day window, not an observed future outcome — it describes what already happened, not what will happen next. That's a reasonable starting point for this notebook, but I plan to move toward a future-window label (prior 90 days of features → next 30 days decline) once I've done the signal audit in a later assignment, so the model predicts something that hasn't happened yet instead of describing something that already has.

In [3]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My metric is **precision@50**: out of the top 50 pages my ranking hands the reviewer, what fraction are truly declining pages? Overall accuracy doesn't matter here, since the reviewer never looks at pages outside the top 50 — a model can be excellent on the bottom 29,950 rows and still be useless if the top 50 are wrong. A simple hand-written rule reaches about 0.24 precision@50 (roughly 12 of 50 correct); a trained model in this repo's pipeline reaches about 0.74 (roughly 37 of 50) — that gap is what I'm trying to close or beat.

In [4]:
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
n = len(df)
print(f"A simple rule flags {declining_with_demand:,} of {n:,} pages "
      f"({100*declining_with_demand/n:.1f}%) -- far more than the reviewer's capacity of 50, "
      f"so precision AT THE TOP of the ranking is what matters, not raw accuracy.")

A simple rule flags 13,152 of 30,000 pages (43.8%) -- far more than the reviewer's capacity of 50, so precision AT THE TOP of the ranking is what matters, not raw accuracy.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = **one content page** (`content_id`), described by its trailing-90-day search and engagement metrics, plus which client it belongs to.

In [5]:
unit_cols = [
    "content_id", "client_id",
    "impressions_90d", "clicks_90d", "ctr", "avg_position",
    "engagement_rate", "content_age_days", "days_since_last_update",
    "trend_direction", "is_declining_label",
]
df[unit_cols].head(10)

,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,engagement_rate,content_age_days,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,5.88,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,0.00,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,0.00,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,1.28,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,0.00,263,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,0.00,147,20,down,1
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,0.00,90,20,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,3.57,445,22,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,5.88,90,20,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,0.00,257,104,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement rule (e.g. "declining AND high impressions") flags 43.8% of all pages as worth reviewing — far too blunt to guide a 50-page-per-cycle reviewer, and it only reaches ~0.24 precision@50 in this repo's baseline. Part of the problem is that there's more than one kind of opportunity signal in this data (declining trend is one; a visible-but-under-clicking page is a separate one, shown below), and a fixed rule can't weigh several correlated, moving signals — position, freshness, CTR, engagement, trend — against each other or learn how they trade off. A learned model reaches ~0.74 precision@50 because it captures those interactions instead of relying on one hard-coded threshold.

In [6]:
low_ctr_visible_page = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0) & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()
print(f"{low_ctr_visible_page:,} pages are visible but under-capturing clicks for their "
      f"position -- a second signal a fixed rule would have to juggle alongside 'declining', "
      f"which is exactly the kind of trade-off ML handles better than an if-statement.")

9,759 pages are visible but under-capturing clicks for their position -- a second signal a fixed rule would have to juggle alongside 'declining', which is exactly the kind of trade-off ML handles better than an if-statement.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.